# Baseline Classifier

Out-of-sample, walk-forward logistic regression predicting the **quantile / binary `label`**
built on the Feature Engineering page. All logic lives in `irp.models.classifier`.

**The dataset comes from `/features`**: build features + a **Quantile** (or Up/Down) label,
then **Export parquet**. This notebook loads that export — it does not rebuild the data.

In [ ]:
from irp.models import classifier as clf
from sklearn.linear_model import LogisticRegression

# ── parameters ──
EXPORT_PATH = None        # None = most recent /features export; or a path/filename
FEATURE_COLS = None       # None = infer (numeric cols except Date/Ticker/fwd_ret/label)
MODEL = LogisticRegression(max_iter=1000)
MIN_TRAIN_DATES = 12
TPL = clf.nb_template()

clf.list_exports()        # available datasets from the /features page

## 1 · Load dataset (must carry a quantile/binary `label`)

In [ ]:
df, features = clf.load_export(EXPORT_PATH, feature_cols=FEATURE_COLS, target='label')
print(len(features), 'features:', features)
df['label'].value_counts().sort_index()

## 2 · Walk-forward classification
Expanding window: fit on the past, predict the current cross-section's class (no look-ahead).

In [ ]:
res = clf.walk_forward_classifier(df, features, target='label', model=MODEL, min_train_dates=MIN_TRAIN_DATES)
_ = clf.summary(res)

## 3 · Visualize
Confusion + accuracy. Quintiles by predicted score need `fwd_ret` in the export (Quantile mode keeps it).

In [ ]:
clf.plot_confusion(res, TPL)

In [ ]:
clf.plot_accuracy(res, TPL)

In [ ]:
clf.plot_quintiles(res, TPL)